# Kaggle Model Factory
Attach CSV via **Add Input**. Run Cell 1 and choose dataset/model, then run Cell 2. For saved Kaggle versions set the defaults in Cell 1 first.

In [ ]:
# CELL 1 — Kaggle Model Factory choices
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

DATASET_INDEX = 0
MODEL_DEFAULT = "Compare tabular models"
TARGET_DEFAULT = "AUTO"
OPTIMIZATION_DEFAULT = "fast"  # fast / balanced
TASK = "auto"                   # auto / classification / regression
EPOCHS = 12
BATCH_SIZE = 64
HIDDEN_UNITS = (128, 64)
LEARNING_RATE = 0.001
MAX_ROWS = 30000
TEST_SIZE = 0.20
SEED = 42
OUTPUT_DIR = "/kaggle/working/model_factory"

MODEL_INFO = {
    "Random Forest": ("random_forest", "General tabular classification and regression. Strong nonlinear baseline for fraud, delays and failure prediction."),
    "Extra Trees": ("extra_trees", "Randomized tree ensembles for classification or regression. Useful for quick benchmarking against Random Forest."),
    "Gradient Boosting": ("hist_gradient_boosting", "Sequential boosted trees for structured prediction. Useful for operational risk and demand forecasting."),
    "Linear / Logistic": ("linear", "Regularized linear regression or classification. A fast, interpretable baseline."),
    "XGBoost": ("xgboost", "Gradient-boosted trees for structured datasets. Often strong for fraud, churn and tabular prediction."),
    "LightGBM": ("lightgbm", "Efficient boosted trees for larger datasets. Useful when training speed and scale matter."),
    "CatBoost": ("catboost", "Boosted trees suited to mixed business features. Especially useful when categorical relationships matter."),
    "PyTorch Neural Network": ("pytorch", "Configurable dense neural network for classification or regression. Useful for learning nonlinear interactions and testing GPU training."),
    "TensorFlow / Keras Neural Network": ("tensorflow", "Configurable dense neural network for classification or regression. Useful for Keras workflows and portable neural model artifacts."),
    "Compare tabular models": ("tabular", "Benchmarks Random Forest, Extra Trees and XGBoost. Good first experiment before deeper tuning."),
    "Compare all including neural networks": ("all", "Benchmarks all available classical and neural model families. More CPU/GPU time and optional dependencies required."),
}
MODEL_GROUPS = {
    "tabular": ["random_forest", "extra_trees", "xgboost"],
    "all": ["random_forest", "extra_trees", "hist_gradient_boosting", "linear", "xgboost", "lightgbm", "catboost", "pytorch", "tensorflow"],
}
paths = sorted(Path("/kaggle/input").rglob("*.csv")) if Path("/kaggle/input").exists() else []
if not paths:
    print("No CSV found. In Kaggle choose Add Input, attach your Dataset or Notebook Output, and rerun Cell 1.")
else:
    dataset_widget = widgets.Dropdown(
        options=[(str(p.relative_to('/kaggle/input')), str(p)) for p in paths],
        index=min(DATASET_INDEX, len(paths)-1), description="Dataset:",
        style={"description_width":"100px"}, layout=widgets.Layout(width="98%"))
    model_widget = widgets.Dropdown(
        options=list(MODEL_INFO), value=MODEL_DEFAULT, description="Model:",
        style={"description_width":"100px"}, layout=widgets.Layout(width="98%"))
    target_widget = widgets.Text(value=TARGET_DEFAULT, description="Target:",
        style={"description_width":"100px"})
    optimization_widget = widgets.Dropdown(options=["fast", "balanced"],
        value=OPTIMIZATION_DEFAULT, description="Optimize:",
        style={"description_width":"100px"})
    help_widget = widgets.HTML()
    def show_help(change=None):
        help_widget.value = "<b>Use case:</b> " + MODEL_INFO[model_widget.value][1]
    model_widget.observe(show_help, names="value")
    show_help()
    display(dataset_widget, model_widget, help_widget, target_widget, optimization_widget)
    print("Change choices above, then run Cell 2. For Save & Run All, also change the DEFAULT values at the top of Cell 1.")


In [ ]:
# CELL 2 — Training, validation and native model exports
import json, time, zipfile, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, ExtraTreesClassifier, ExtraTreesRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error, r2_score

if not paths:
    raise RuntimeError("No CSVs attached. Use Kaggle Add Input, then rerun Cell 1.")
source = dataset_widget.value
choice = MODEL_INFO[model_widget.value][0]
models = MODEL_GROUPS.get(choice, [choice])
target_requested = target_widget.value.strip()
optimize = optimization_widget.value
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

# Our industry dataset generator's label registry.
known = {
    "flight_delays":"delay_minutes", "maintenance":"needs_maintenance", "transaction_fraud":"is_fraud",
    "loan_default":"loan_default", "appointment_no_show":"no_show", "readmission_risk":"readmitted_30d",
    "customer_churn":"churned", "demand_forecast":"units_demanded", "equipment_failure":"failure_next_7d",
    "quality_defect":"defect_found", "late_delivery":"late_delivery", "shipment_damage":"damaged",
    "suspicious_login":"suspicious_login", "alert_severity":"severity_level", "high_latency":"high_latency",
    "capacity_forecast":"requests_next_hour", "peak_load":"demand_kw", "equipment_outage":"outage_next_30d",
    "property_price":"price_usd", "long_listing":"listing_over_90d"
}
regression_labels = {"delay_minutes", "units_demanded", "requests_next_hour", "demand_kw", "price_usd", "latency_ms"}
df = pd.read_csv(source)
if df.empty: raise ValueError("Dataset is empty")
if df.columns.duplicated().any(): raise ValueError("Duplicate column names")
if len(df) > MAX_ROWS: df = df.sample(MAX_ROWS, random_state=SEED)
if target_requested.upper() == "AUTO":
    matches = list(dict.fromkeys(v for k,v in known.items() if k in Path(source).stem.lower() and v in df))
    if not matches: matches = list(dict.fromkeys(v for v in known.values() if v in df))
    if len(matches) != 1: raise ValueError(f"Set Target explicitly in Cell 1; candidates: {matches}")
    target = matches[0]
else:
    target = target_requested
if target not in df: raise ValueError(f"Missing target column {target}")
df = df.dropna(subset=[target]).copy()
task = ("regression" if target in regression_labels else "classification") if TASK == "auto" else TASK
if task not in ("regression", "classification"): raise ValueError("TASK must be auto/classification/regression")
excluded = {target} | {c for c in df if c.lower() in ("id", "uuid", "record_id") or c.lower().endswith("_id")}
excluded |= {c for c in df if c != target and c in known.values()}
if target == "high_latency": excluded.add("latency_ms")  # direct leakage from generator
X = df.drop(columns=list(excluded), errors="ignore").copy()
y = df[target].copy()
label_encoder = None
if task == "classification":
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y.astype(str))
    if len(label_encoder.classes_) < 2: raise ValueError("Classification needs at least two classes")
else:
    y = pd.to_numeric(y, errors="coerce")
    good = y.notna(); X = X.loc[good]; y = y.loc[good]

numeric = X.select_dtypes(include="number").columns.tolist()
text_cols = [c for c in X if c not in numeric and any(s in c.lower() for s in ("note", "description", "feedback", "comment", "narrative", "report", "log", "text", "event"))]
cat_cols = [c for c in X if c not in numeric + text_cols]
# Ensure text fields have no nulls and remain usable by TF-IDF inside saved pipeline.
for c in text_cols: X[c] = X[c].fillna("").astype(str)
for c in cat_cols: X[c] = X[c].fillna("missing").astype(str)
large_cat = [c for c in cat_cols if X[c].nunique() > 300]
X = X.drop(columns=large_cat); cat_cols = [c for c in cat_cols if c not in large_cat]
parts = []
if numeric:
    parts.append(("num", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric))
if cat_cols:
    parts.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))
for i,c in enumerate(text_cols):
    parts.append((f"text_{i}", TfidfVectorizer(max_features=200, ngram_range=(1,2)), c))
if not parts: raise ValueError("No usable features")
preprocessor = ColumnTransformer(parts, sparse_threshold=0)
counts = pd.Series(y).value_counts() if task == "classification" else None
strat = y if task == "classification" and counts.min() >= 2 and int(np.ceil(len(y)*TEST_SIZE)) >= len(counts) else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=strat)
if task == "classification":
    n_folds = min(CV_FOLDS, int(pd.Series(y_train).value_counts().min()))
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    metric = "f1_weighted"
else:
    n_folds = min(CV_FOLDS, len(y_train))
    cv = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    metric = "neg_mean_absolute_error"
if n_folds < 2: raise ValueError("Not enough records per class for cross-validation")

def classical(name):
    reg = task == "regression"
    if name == "random_forest":
        c = RandomForestRegressor if reg else RandomForestClassifier
        return c(n_estimators=120, random_state=SEED, n_jobs=-1), {"model__max_depth":[None,8,16]}
    if name == "extra_trees":
        c = ExtraTreesRegressor if reg else ExtraTreesClassifier
        return c(n_estimators=120, random_state=SEED, n_jobs=-1), {"model__max_depth":[None,8,16]}
    if name == "hist_gradient_boosting":
        c = HistGradientBoostingRegressor if reg else HistGradientBoostingClassifier
        return c(random_state=SEED), {"model__learning_rate":[0.03,0.1], "model__max_iter":[100,200]}
    if name == "linear":
        return (Ridge(), {"model__alpha":[0.1,1,10]}) if reg else (LogisticRegression(max_iter=1500), {"model__C":[0.1,1,10]})
    if name == "xgboost":
        from xgboost import XGBRegressor, XGBClassifier
        c = XGBRegressor if reg else XGBClassifier
        return c(n_estimators=120, tree_method="hist", random_state=SEED, n_jobs=2), {"model__max_depth":[3,5,8]}
    if name == "lightgbm":
        from lightgbm import LGBMRegressor, LGBMClassifier
        c = LGBMRegressor if reg else LGBMClassifier
        return c(n_estimators=120, verbose=-1, random_state=SEED, n_jobs=2), {"model__num_leaves":[15,31,63]}
    if name == "catboost":
        from catboost import CatBoostRegressor, CatBoostClassifier
        c = CatBoostRegressor if reg else CatBoostClassifier
        return c(iterations=120, verbose=False, random_seed=SEED), {"model__depth":[4,6,8]}
    raise ValueError(name)

# For neural networks, fit preprocessing on training data only; then export it separately.
# Both networks share the same dense input matrix and architecture settings.
def neural_arrays():
    from sklearn.base import clone
    proc = clone(preprocessor)
    a = np.asarray(proc.fit_transform(X_train), dtype=np.float32)
    b = np.asarray(proc.transform(X_test), dtype=np.float32)
    return proc, np.nan_to_num(a), np.nan_to_num(b)

results, exported, failures = [], {}, {}
print("DATASET:", source, "\nTARGET:", target, "\nTASK:", task, "\nMODELS:", models)
for name in models:
    started = time.time()
    try:
        if name not in ("pytorch", "tensorflow"):
            estimator, grid = classical(name)
            pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
            if optimize == "balanced":
                search = RandomizedSearchCV(pipe, grid, n_iter=min(5, int(np.prod([len(v) for v in grid.values()]))), cv=cv, scoring=metric, n_jobs=1, random_state=SEED, error_score="raise")
                search.fit(X_train, y_train)
                fitted, cv_score = search.best_estimator_, float(search.best_score_)
            else:
                cv_score = float(cross_val_score(pipe, X_train, y_train, scoring=metric, cv=cv, n_jobs=1).mean())
                fitted = pipe.fit(X_train, y_train)
            predictions = fitted.predict(X_test)
            artifact = out / f"{name}_pipeline.joblib"
            joblib.dump(fitted, artifact)
            exported[name] = [artifact]
        elif name == "pytorch":
            import torch
            from torch import nn
            from torch.utils.data import DataLoader, TensorDataset
            torch.manual_seed(SEED)
            proc, a, b = neural_arrays()
            n_out = 1 if task == "regression" else len(label_encoder.classes_)
            dims = [a.shape[1], *HIDDEN_UNITS, n_out]
            layers = []
            for i in range(len(dims)-2): layers += [nn.Linear(dims[i],dims[i+1]), nn.ReLU()]
            layers += [nn.Linear(dims[-2],dims[-1])]
            net = nn.Sequential(*layers)
            device = "cuda" if torch.cuda.is_available() else "cpu"
            net = net.to(device)
            optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
            loss_fn = nn.MSELoss() if task == "regression" else nn.CrossEntropyLoss()
            labels = torch.tensor(np.asarray(y_train), dtype=torch.float32 if task == "regression" else torch.long)
            loader = DataLoader(TensorDataset(torch.tensor(a), labels), batch_size=BATCH_SIZE, shuffle=True)
            for epoch in range(EPOCHS):
                net.train()
                for xb,yb in loader:
                    xb,yb = xb.to(device),yb.to(device)
                    optimizer.zero_grad()
                    z = net(xb)
                    loss = loss_fn(z.squeeze(-1), yb) if task == "regression" else loss_fn(z,yb)
                    loss.backward(); optimizer.step()
            net.eval()
            with torch.no_grad():
                logits = net(torch.tensor(b).to(device)).cpu().numpy()
            predictions = logits.ravel() if task == "regression" else logits.argmax(axis=1)
            checkpoint = out / "pytorch_model.pt"
            torch.save({"state_dict":net.cpu().state_dict(), "dimensions":dims, "task":task, "classes":None if label_encoder is None else label_encoder.classes_.tolist()}, checkpoint)
            prep_file = out / "pytorch_preprocessor.joblib"; joblib.dump(proc, prep_file)
            exported[name] = [checkpoint, prep_file]
            cv_score = np.nan  # neural models evaluated on untouched holdout (not CV)
        else:
            import tensorflow as tf
            tf.keras.utils.set_random_seed(SEED)
            proc, a, b = neural_arrays()
            n_out = 1 if task == "regression" else len(label_encoder.classes_)
            net = tf.keras.Sequential([tf.keras.layers.Input(shape=(a.shape[1],)), *[tf.keras.layers.Dense(h, activation="relu") for h in HIDDEN_UNITS], tf.keras.layers.Dense(n_out)])
            net.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="mse" if task == "regression" else tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
            net.fit(a, np.asarray(y_train), epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.15, verbose=0)
            logits = net.predict(b, verbose=0)
            predictions = logits.ravel() if task == "regression" else logits.argmax(axis=1)
            artifact = out / "tensorflow_model.keras"; net.save(str(artifact))
            prep_file = out / "tensorflow_preprocessor.joblib"; joblib.dump(proc, prep_file)
            exported[name] = [artifact, prep_file]
            cv_score = np.nan
        value = float(mean_absolute_error(y_test,predictions)) if task == "regression" else float(f1_score(y_test,predictions,average="weighted",zero_division=0))
        results.append({"model":name, "holdout_metric":value, "cv_score":cv_score, "seconds":round(time.time()-started,2)})
        print(name, "OK", "MAE" if task == "regression" else "F1", round(value,4))
    except Exception as exc:
        failures[name] = f"{type(exc).__name__}: {exc}"
        print(name, "SKIPPED:", failures[name])

if not results: raise RuntimeError(f"No models trained. Errors: {failures}")
board = pd.DataFrame(results).sort_values("holdout_metric", ascending=(task=="regression")).reset_index(drop=True)
display(board)
board.to_csv(out / "leaderboard.csv",index=False)
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(board.model,board.holdout_metric)
ax.set_title("Holdout MAE (lower better)" if task=="regression" else "Holdout F1 (higher better)")
ax.tick_params(axis="x",rotation=40)
plt.tight_layout(); fig.savefig(out / "benchmark.png",dpi=120); plt.show(); plt.close(fig)

# All trained model artifacts are retained; winner is descriptive only.
champion = board.iloc[0]["model"]
X_test.head(5).to_csv(out / "sample_input.csv",index=False)
manifest = {"source":source,"target":target,"task":task,"champion_by_holdout":champion,
 "feature_columns":X.columns.tolist(),"classes":None if label_encoder is None else label_encoder.classes_.tolist(),
 "pytorch_layers":list(HIDDEN_UNITS),"epochs":EPOCHS,"seed":SEED,"python":platform.python_version(),
 "artifacts":{k:[p.name for p in v] for k,v in exported.items()},"failures":failures,
 "notes":"Holdout scores are exploratory; selecting a winner on holdout biases its estimate. Synthetic data does not prove real-world performance. Load joblib only from trusted sources."}
(out / "manifest.json").write_text(json.dumps(manifest,indent=2,default=str))
zip_path = out.parent / "model_factory_artifacts.zip"
zip_path.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for p in out.iterdir():
        if p.is_file(): archive.write(p,arcname=p.name)
print("\nBEST HOLDOUT:",champion,"\nARTIFACTS:",out,"\nZIP:",zip_path)
print("Kaggle: Save Version / Save & Run All to retain outputs; create a Kaggle Model from those outputs separately.")
